In [1]:
print('test')

test


In [39]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI 
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.messages import HumanMessage
import asyncio
from IPython.display import Markdown
from langchain_community.chat_models import ChatOllama

In [47]:
load_dotenv(override=True)

True

In [48]:
mcp_client = MultiServerMCPClient(
    {
        "mcpserver": {
        "transport": "streamable_http",
        "url": "http://localhost:24000/mcp",
        }
    }
)

In [49]:
tools = await mcp_client.get_tools()

In [50]:
tools

[StructuredTool(name='get_employee_infos', description='\n    Get information about an employee.\n    ', args_schema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_employee_infosArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x752dd4fee7a0>),
 StructuredTool(name='search', description='\n    Search the web for the given query.\n    ', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'searchArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x752dd4fee8e0>)]

In [51]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [52]:
agent = create_agent(
    model=llm, tools=tools, system_prompt="answer user question using provided tools"
    
)

In [33]:
# Forcez l'utilisation de la clé DeepSeek
deepseek_key = os.getenv("DEEPSEEK_API_KEY")

# Nettoyez toute variable d'environnement OpenAI existante
if "OPENAI_API_KEY" in os.environ:
    del os.environ["OPENAI_API_KEY"]

# Créez le LLM avec la clé explicite
llm = ChatOpenAI(
    model="deepseek-chat",
    temperature=0.7,
    api_key=deepseek_key,  # Forcez la clé DeepSeek
    base_url="https://api.deepseek.com/v1",
)

In [53]:
resp = await agent.ainvoke({
    "messages": [
        HumanMessage("Quel est le salaire de Rahma?")
    ]

})

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************pWMA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [ ]:
print(resp['messages'][-1].content)

Le produit intérieur brut (PIB) du Maroc a atteint 1.463,3 milliards de dirhams (MMDH) en 2023, ce qui représente une hausse de 10% par rapport à l'année précédente.


In [ ]:
resp = await agent.ainvoke({
    "messages": [
        HumanMessage("Quel est le PIB du Maroc aujourd'hui?")
    ]

})

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
print(display(Markdown(resp['messages'][-1].content)))

Le produit intérieur brut (PIB) du Maroc a atteint 1.463,3 milliards de dirhams (MMDH) en 2023, ce qui représente une hausse de 10% par rapport à l'année précédente.

None
